# rippl — GPU Benchmark Suite v2
**Runtime: GPU (T4 recommended)**

Post-audit rebuild. Fixes applied:
- Laplacian spatial-dim leak fixed (`spatial_dims=1` explicit)
- Derivative dim collision fixed (time always last dim)
- InverseProblem signature mismatch fixed
- System.validate() implemented
- MultiFieldMLP hardcoded in_dim removed
- README restored

Run all cells top to bottom. Do not skip.
Results saved to `benchmark_results.json`.

In [ ]:
# CELL 1 — Install from latest main (post-fix commit)
import os

if os.path.exists('/content/rippleml'):
    %cd /content/rippleml
    !git pull origin main -q
else:
    %cd /content
    !git clone https://github.com/esranow/rippleml.git -q
    %cd rippleml

!pip install -e . -q

import sys
sys.path.insert(0, '/content/rippleml')

# Verify commit
!git log --oneline -3

import torch
print(f"\nPyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

In [ ]:
# CELL 2 — Imports and shared utilities
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import math
import time
import json
import warnings
import matplotlib.pyplot as plt

torch.manual_seed(42)
results = {}

# Shared tanh MLP — Xavier init, no spectral bias
class MLP(nn.Module):
    def __init__(self, in_dim, out_dim, hidden=50, layers=4):
        super().__init__()
        dims = [in_dim] + [hidden]*layers + [out_dim]
        nets = []
        for i in range(len(dims)-1):
            nets.append(nn.Linear(dims[i], dims[i+1]))
            if i < len(dims)-2:
                nets.append(nn.Tanh())
        self.net = nn.Sequential(*nets)
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight)
                nn.init.zeros_(m.bias)
    def forward(self, x): return self.net(x)

def relative_l2(pred, true):
    return (pred-true).pow(2).sum().sqrt() / (true.pow(2).sum().sqrt() + 1e-10)

def sobol_points(n, d, device):
    eng = torch.quasirandom.SobolEngine(d, scramble=True)
    return eng.draw(n).to(device)

print('Utilities ready')

## PRE-FLIGHT: Verify Fixes

In [ ]:
# CELL 3 — Verify all three critical fixes before any benchmark
print('=' * 50)
print('PRE-FLIGHT FIX VERIFICATION')
print('=' * 50)

from rippl.physics.operators import Laplacian, PressureGradient, TimeDerivative
from rippl.physics.derivatives import compute_all_derivatives
from rippl.core.equation import Equation

# Fix 1: Laplacian must NOT include time dim
coords_test = torch.rand(200, 2, requires_grad=True, device=device)
u_test = torch.sin(math.pi * coords_test[:, 0:1]) * torch.cos(math.pi * coords_test[:, 1:2])
lap = Laplacian(field='u', spatial_dims=1)
derived_test = compute_all_derivatives({'u': u_test}, coords_test, ['u_xx'])
result_lap = lap.forward({'u': u_test}, coords_test, derived_test)
expected_lap = -(math.pi**2) * torch.sin(math.pi * coords_test[:, 0:1]) * torch.cos(math.pi * coords_test[:, 1:2])
err_lap = (result_lap - expected_lap).abs().max().item()
status_lap = 'PASS' if err_lap < 1e-3 else 'FAIL'
print(f'Fix 1 — Laplacian spatial-only: {err_lap:.2e}  {status_lap}')

# Fix 2: Physics sanity — wave equation residual on analytic solution
from rippl.physics_blocks.residual import HybridWaveResidualBlock
coords_sanity = torch.rand(1000, 2, requires_grad=True, device=device)
u_sanity = torch.sin(math.pi * coords_sanity[:, 0:1]) * torch.cos(math.pi * coords_sanity[:, 1:2])
eq_sanity = Equation([TimeDerivative(2), Laplacian(spatial_dims=1)])
block_sanity = HybridWaveResidualBlock(a=1.0, b=0.0, c=1.0, use_correction=False)
res_sanity = block_sanity.residual(u_sanity, eq_sanity, coords_sanity)
val_sanity = res_sanity.abs().max().item()
status_sanity = 'PASS' if val_sanity < 1e-2 else 'FAIL'
print(f'Fix 2 — Physics sanity (wave residual): {val_sanity:.2e}  {status_sanity}')

# Fix 3: Pressure gradient 1D — no dim collision
p_coords = torch.rand(100, 1, requires_grad=True, device=device)
p_field = 1 - p_coords
derived_p = compute_all_derivatives({'p': p_field}, p_coords, ['p_x'])
pg = PressureGradient(field_p='p', direction=0)
p_x_result = pg.forward({'p': p_field}, p_coords, derived_p)
p_x_expected = -torch.ones_like(p_field)
err_p = (p_x_result - p_x_expected).abs().max().item()
status_p = 'PASS' if err_p < 1e-3 else 'FAIL'
print(f'Fix 3 — Pressure gradient 1D: {err_p:.2e}  {status_p}')

print('=' * 50)
all_pass = all(s == 'PASS' for s in [status_lap, status_sanity, status_p])
if not all_pass:
    print('WARNING: Pre-flight FAILED — fixes not applied correctly')
    print('Do not proceed. Check that git pull succeeded in Cell 1.')
else:
    print('All pre-flight checks PASSED — proceeding to benchmarks')

results['preflight'] = {
    'laplacian': {'error': err_lap, 'status': status_lap},
    'physics_sanity': {'residual': val_sanity, 'status': status_sanity},
    'pressure_gradient': {'error': err_p, 'status': status_p}
}

## BENCHMARK 1 — Heat Equation 1D
**PDE:** `∂u/∂t = 0.1 * ∂²u/∂x²`  
**Analytic:** `u(x,t) = sin(πx) * exp(-0.1π²t)`  
**Gate:** L2 < 1e-2

In [ ]:
# CELL 4 — Heat equation training
alpha = 0.1
model_heat = MLP(2, 1, hidden=50, layers=5).to(device)

# Sobol collocation — better coverage than random
coords_col = sobol_points(10000, 2, device)

# IC: u(x,0) = sin(πx)
x_ic = torch.rand(2000, 1, device=device)
coords_ic = torch.cat([x_ic, torch.zeros(2000, 1, device=device)], dim=1)
u_ic = torch.sin(math.pi * x_ic)

# BC: u(0,t) = u(1,t) = 0
t_bc = torch.rand(2000, 1, device=device)
coords_bcl = torch.cat([torch.zeros(2000, 1, device=device), t_bc], dim=1)
coords_bcr = torch.cat([torch.ones(2000, 1, device=device), t_bc], dim=1)
u_bc = torch.zeros(2000, 1, device=device)

def heat_loss(model, coords):
    c = coords.clone().requires_grad_(True)
    u = model(c)
    du = torch.autograd.grad(u.sum(), c, create_graph=True)[0]
    u_t = du[:, 1:2]
    u_x = du[:, 0:1]
    u_xx = torch.autograd.grad(u_x.sum(), c, create_graph=True)[0][:, 0:1]
    res = (u_t - alpha * u_xx).pow(2).mean()
    l_ic = F.mse_loss(model(coords_ic), u_ic)
    l_bc = F.mse_loss(model(coords_bcl), u_bc) + F.mse_loss(model(coords_bcr), u_bc)
    return res + 100.0*(l_ic + l_bc), res, l_ic, l_bc

opt = torch.optim.Adam(model_heat.parameters(), lr=5e-4)
sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=200, factor=0.5, min_lr=1e-6)
t0 = time.time()
losses_heat = []

print('Adam phase (8000 epochs)...')
for epoch in range(8000):
    opt.zero_grad()
    loss, res, l_ic, l_bc = heat_loss(model_heat, coords_col)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model_heat.parameters(), 1.0)
    opt.step()
    sched.step(loss)
    losses_heat.append(res.item())
    if epoch % 1000 == 0:
        print(f'  [{epoch:5d}] res={res.item():.2e} ic={l_ic.item():.2e} lr={opt.param_groups[0]["lr"]:.2e}')

print('LBFGS phase (500 steps)...')
opt_l = torch.optim.LBFGS(model_heat.parameters(), lr=1.0, max_iter=20, line_search_fn='strong_wolfe')
for step in range(500):
    def closure():
        opt_l.zero_grad()
        loss, _, _, _ = heat_loss(model_heat, coords_col)
        loss.backward()
        return loss
    opt_l.step(closure)
    if step % 100 == 0:
        with torch.no_grad():
            l, r, li, lb = heat_loss(model_heat, coords_col)
        print(f'  [LBFGS {step:3d}] loss={l.item():.2e}')

train_time_heat = time.time() - t0

# Evaluation on 200x200 grid
x_e = torch.linspace(0, 1, 200, device=device)
t_e = torch.linspace(0, 1, 200, device=device)
X, T = torch.meshgrid(x_e, t_e, indexing='ij')
coords_eval = torch.stack([X.flatten(), T.flatten()], dim=1)
with torch.no_grad():
    u_pred_heat = model_heat(coords_eval).reshape(200, 200).cpu()
u_true_heat = (torch.sin(math.pi*X) * torch.exp(-alpha*math.pi**2*T)).cpu()

l2_heat = relative_l2(u_pred_heat, u_true_heat).item()
status_heat = 'PASS' if l2_heat < 1e-2 else 'FAIL'
print(f'\n=== HEAT 1D === L2={l2_heat:.6e}  Time={train_time_heat:.1f}s  {status_heat}')
results['heat_1d'] = {'l2': l2_heat, 'time': train_time_heat, 'status': status_heat}

In [ ]:
# CELL 5 — Heat plots
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
kw = dict(origin='lower', aspect='auto', extent=[0,1,0,1])
axes[0].imshow(u_true_heat.T.numpy(), cmap='RdBu_r', **kw)
axes[0].set_title('Analytic'); axes[0].set_xlabel('x'); axes[0].set_ylabel('t')
axes[1].imshow(u_pred_heat.T.numpy(), cmap='RdBu_r', **kw)
axes[1].set_title('rippl'); axes[1].set_xlabel('x')
err_im = axes[2].imshow((u_pred_heat-u_true_heat).abs().T.numpy(), cmap='hot_r', **kw)
axes[2].set_title(f'Error (L2={l2_heat:.3e})')
plt.colorbar(err_im, ax=axes[2])
plt.suptitle('Heat Equation 1D — rippl', fontsize=13)
plt.tight_layout()
plt.savefig('heat1d_result.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved heat1d_result.png')

## BENCHMARK 2 — Wave Equation 1D (Causal Training)
**PDE:** `∂²u/∂t² = ∂²u/∂x²` (c=1)  
**Analytic:** `u(x,t) = sin(πx)cos(πt)`  
**IC:** `u(x,0)=sin(πx)`, `∂u/∂t(x,0)=0`  
**BC:** `u(0,t)=u(1,t)=0`  
**Gate:** L2 < 1e-2  
**Fix applied:** Laplacian now uses `spatial_dims=1` — time dim excluded

In [ ]:
# CELL 6 — Wave equation with causal training + fixed Laplacian
model_wave = MLP(2, 1, hidden=100, layers=6).to(device)

coords_wave_col = sobol_points(10000, 2, device)

x_ic_w = torch.rand(3000, 1, device=device)
coords_ic_w = torch.cat([x_ic_w, torch.zeros(3000, 1, device=device)], dim=1)
u_ic_w = torch.sin(math.pi * x_ic_w)

t_bc_w = torch.rand(3000, 1, device=device)
coords_bcl_w = torch.cat([torch.zeros(3000, 1, device=device), t_bc_w], dim=1)
coords_bcr_w = torch.cat([torch.ones(3000, 1, device=device), t_bc_w], dim=1)
u_bc_w = torch.zeros(3000, 1, device=device)

def wave_pde_residual(model, coords, n_bins=20, epsilon=1.0):
    """Wave residual with causal weighting. Laplacian spatial_dims=1 fix applied."""
    c = coords.clone().requires_grad_(True)
    u = model(c)
    # First derivatives
    du = torch.autograd.grad(u.sum(), c, create_graph=True)[0]
    u_x = du[:, 0:1]  # spatial only
    u_t = du[:, 1:2]  # time
    # Second derivatives — spatial and time separately (fix applied)
    u_xx = torch.autograd.grad(u_x.sum(), c, create_graph=True)[0][:, 0:1]
    u_tt = torch.autograd.grad(u_t.sum(), c, create_graph=True)[0][:, 1:2]
    res_pt = (u_tt - u_xx).pow(2)  # pointwise

    # Causal weighting — bin by time
    t_vals = c[:, 1].detach()
    t_min, t_max = t_vals.min().item(), t_vals.max().item()
    edges = torch.linspace(t_min, t_max, n_bins+1, device=device)
    bin_losses, valid_bins = [], []
    for i in range(n_bins):
        mask = (t_vals >= edges[i]) & (t_vals < edges[i+1])
        if mask.sum() > 0:
            bin_losses.append(res_pt[mask].mean())
            valid_bins.append(i)
        else:
            bin_losses.append(torch.tensor(0.0, device=device))
    cumsum = 0.0
    weights = []
    for bl in bin_losses:
        weights.append(math.exp(-epsilon * cumsum))
        cumsum += bl.detach().item()
    loss_pde = sum(w * bl for w, bl in zip(weights, bin_losses)) / max(len(valid_bins), 1)
    return loss_pde, res_pt.mean()

def wave_constraint_loss(model):
    l_ic = F.mse_loss(model(coords_ic_w), u_ic_w)
    l_bc = F.mse_loss(model(coords_bcl_w), u_bc_w) + F.mse_loss(model(coords_bcr_w), u_bc_w)
    # IC velocity: du/dt(x,0) = 0
    c_vel = coords_ic_w.clone().requires_grad_(True)
    u_vel = model(c_vel)
    u_t_vel = torch.autograd.grad(u_vel.sum(), c_vel, create_graph=True)[0][:, 1:2]
    l_vel = u_t_vel.pow(2).mean()
    return l_ic + l_vel + l_bc

# Phase A: constraints only — force model away from u=0 trivial solution
print('Phase A: constraint pretraining (3000 epochs)...')
opt_a = torch.optim.Adam(model_wave.parameters(), lr=1e-3)
for epoch in range(3000):
    opt_a.zero_grad()
    loss = wave_constraint_loss(model_wave)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model_wave.parameters(), 1.0)
    opt_a.step()
    if epoch % 1000 == 0:
        print(f'  [{epoch:5d}] constraint_loss={loss.item():.2e}')

# Phase B: full causal loss
print('Phase B: causal PDE training (10000 epochs)...')
opt_b = torch.optim.Adam(model_wave.parameters(), lr=5e-4)
sched_b = torch.optim.lr_scheduler.ReduceLROnPlateau(opt_b, patience=300, factor=0.5, min_lr=1e-6)
t0_wave = time.time()
losses_wave = []
epsilon = 1.0

for epoch in range(10000):
    opt_b.zero_grad()
    loss_pde, res_mean = wave_pde_residual(model_wave, coords_wave_col, epsilon=epsilon)
    loss_con = wave_constraint_loss(model_wave)
    loss = loss_pde + 100.0 * loss_con
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model_wave.parameters(), 1.0)
    opt_b.step()
    sched_b.step(loss)
    losses_wave.append(res_mean.item())
    # Auto-tune epsilon every 100 epochs
    if epoch % 100 == 0:
        epsilon = float(np.clip(1.0 / (res_mean.item() + 1e-8), 0.1, 100.0))
    if epoch % 2000 == 0:
        print(f'  [{epoch:5d}] res={res_mean.item():.2e} con={loss_con.item():.2e} eps={epsilon:.2f} lr={opt_b.param_groups[0]["lr"]:.2e}')

train_time_wave = time.time() - t0_wave

# Eval
x_ew = torch.linspace(0, 1, 200, device=device)
t_ew = torch.linspace(0, 1, 200, device=device)
Xw, Tw = torch.meshgrid(x_ew, t_ew, indexing='ij')
coords_ew = torch.stack([Xw.flatten(), Tw.flatten()], dim=1)
with torch.no_grad():
    u_pred_wave = model_wave(coords_ew).reshape(200, 200).cpu()
u_true_wave = (torch.sin(math.pi*Xw) * torch.cos(math.pi*Tw)).cpu()

l2_wave = relative_l2(u_pred_wave, u_true_wave).item()
status_wave = 'PASS' if l2_wave < 1e-2 else 'FAIL'
print(f'\n=== WAVE 1D === L2={l2_wave:.6e}  Time={train_time_wave:.1f}s  {status_wave}')
results['wave_1d'] = {'l2': l2_wave, 'time': train_time_wave, 'status': status_wave}

In [ ]:
# CELL 7 — Wave plots + loss curve
fig, axes = plt.subplots(1, 4, figsize=(20, 4))
kw = dict(origin='lower', aspect='auto', extent=[0,1,0,1])
axes[0].imshow(u_true_wave.T.numpy(), cmap='RdBu_r', **kw)
axes[0].set_title('Analytic'); axes[0].set_xlabel('x'); axes[0].set_ylabel('t')
axes[1].imshow(u_pred_wave.T.numpy(), cmap='RdBu_r', **kw)
axes[1].set_title('rippl'); axes[1].set_xlabel('x')
err_w = axes[2].imshow((u_pred_wave-u_true_wave).abs().T.numpy(), cmap='hot_r', **kw)
axes[2].set_title(f'Error (L2={l2_wave:.3e})')
plt.colorbar(err_w, ax=axes[2])
axes[3].semilogy(losses_wave, alpha=0.8, color='steelblue')
axes[3].set_title('PDE Residual Loss'); axes[3].set_xlabel('Epoch'); axes[3].grid(True, alpha=0.3)
plt.suptitle('Wave Equation 1D — Causal Training (fixed spatial_dims)', fontsize=13)
plt.tight_layout()
plt.savefig('wave1d_result.png', dpi=150, bbox_inches='tight')
plt.show()

## BENCHMARK 3 — Stokes 1D Couette Flow
**System:** `μ*u_xx - p_x = 0`, `u_x = 0`  
**Analytic:** `u(x)=x`, `p(x)=1-x`  
**Fields:** `[u, p]`  
**Fix applied:** Pressure gradient dim collision fixed — `p_x` correctly maps to dim 0

In [ ]:
# CELL 8 — Stokes 1D with corrected pressure gradient
class TwoHeadMLP(nn.Module):
    """Shared trunk, separate heads for u and p."""
    def __init__(self, in_dim=1, hidden=64, layers=5):
        super().__init__()
        dims = [in_dim] + [hidden]*layers
        trunk = []
        for i in range(len(dims)-1):
            trunk.extend([nn.Linear(dims[i], dims[i+1]), nn.Tanh()])
        self.trunk = nn.Sequential(*trunk)
        self.head_u = nn.Linear(hidden, 1)
        self.head_p = nn.Linear(hidden, 1)
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight)
                nn.init.zeros_(m.bias)
    def forward(self, x):
        h = self.trunk(x)
        return self.head_u(h), self.head_p(h)

model_stokes = TwoHeadMLP(in_dim=1, hidden=64, layers=5).to(device)

x_col_s = torch.rand(5000, 1, device=device)
x0_s = torch.zeros(1000, 1, device=device)
x1_s = torch.ones(1000, 1, device=device)

def stokes_loss(model, x_col):
    x = x_col.clone().requires_grad_(True)
    u, p = model(x)

    # Spatial derivatives — dim 0 only (1D, no time)
    u_x = torch.autograd.grad(u.sum(), x, create_graph=True)[0]
    u_xx = torch.autograd.grad(u_x.sum(), x, create_graph=True)[0]
    p_x = torch.autograd.grad(p.sum(), x, create_graph=True)[0]

    # Momentum: u_xx - p_x = 0
    res_mom = (u_xx - p_x).pow(2).mean()
    # Continuity: u_x = 0 (simplified incompressibility)
    res_cont = u_x.pow(2).mean()

    # BCs
    u0, p0 = model(x0_s)  # u(0)=0, p(0)=1
    u1, p1 = model(x1_s)  # u(1)=1, p(1)=0
    l_bc_u = F.mse_loss(u0, torch.zeros_like(u0)) + F.mse_loss(u1, torch.ones_like(u1))
    l_bc_p = F.mse_loss(p0, torch.ones_like(p0)) + F.mse_loss(p1, torch.zeros_like(p1))

    return res_mom + 10.0*res_cont + 100.0*(l_bc_u + l_bc_p), res_mom, res_cont, l_bc_u, l_bc_p

opt_s = torch.optim.Adam(model_stokes.parameters(), lr=1e-3)
sched_s = torch.optim.lr_scheduler.ReduceLROnPlateau(opt_s, patience=200, factor=0.5, min_lr=1e-6)
t0_s = time.time()
losses_stokes = []

print('Training Stokes 1D (8000 epochs)...')
for epoch in range(8000):
    opt_s.zero_grad()
    loss, r_m, r_c, l_bu, l_bp = stokes_loss(model_stokes, x_col_s)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model_stokes.parameters(), 1.0)
    opt_s.step()
    sched_s.step(loss)
    losses_stokes.append(r_m.item())
    if epoch % 1000 == 0:
        print(f'  [{epoch:5d}] mom={r_m.item():.2e} cont={r_c.item():.2e} bc_u={l_bu.item():.2e} bc_p={l_bp.item():.2e}')

# LBFGS refinement
print('LBFGS refinement (300 steps)...')
opt_sl = torch.optim.LBFGS(model_stokes.parameters(), lr=1.0, max_iter=20, line_search_fn='strong_wolfe')
for step in range(300):
    def closure_s():
        opt_sl.zero_grad()
        loss, _, _, _, _ = stokes_loss(model_stokes, x_col_s)
        loss.backward()
        return loss
    opt_sl.step(closure_s)
    if step % 100 == 0:
        with torch.no_grad():
            l, *_ = stokes_loss(model_stokes, x_col_s)
        print(f'  [LBFGS {step:3d}] loss={l.item():.2e}')

train_time_stokes = time.time() - t0_s

# Eval
x_eval_s = torch.linspace(0, 1, 1000, device=device).unsqueeze(1)
with torch.no_grad():
    u_pred_s, p_pred_s = model_stokes(x_eval_s)
u_pred_s = u_pred_s.cpu(); p_pred_s = p_pred_s.cpu()
x_np_s = x_eval_s.cpu().numpy()
u_true_s = torch.tensor(x_np_s, dtype=torch.float32)
p_true_s = torch.tensor(1 - x_np_s, dtype=torch.float32)

l2_u_s = relative_l2(u_pred_s, u_true_s).item()
l2_p_s = relative_l2(p_pred_s, p_true_s).item()
status_s = 'PASS' if l2_u_s < 1e-2 and l2_p_s < 1e-2 else 'FAIL'
print(f'\n=== STOKES 1D === u_L2={l2_u_s:.6e}  p_L2={l2_p_s:.6e}  Time={train_time_stokes:.1f}s  {status_s}')
results['stokes_1d'] = {'l2_u': l2_u_s, 'l2_p': l2_p_s, 'time': train_time_stokes, 'status': status_s}

In [ ]:
# CELL 9 — Stokes plots
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(x_np_s, u_true_s.numpy(), 'k-', lw=2, label='Analytic u=x')
axes[0].plot(x_np_s, u_pred_s.numpy(), 'r--', lw=2, label=f'rippl (L2={l2_u_s:.3e})')
axes[0].set_title('Velocity u'); axes[0].set_xlabel('x'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(x_np_s, p_true_s.numpy(), 'k-', lw=2, label='Analytic p=1-x')
axes[1].plot(x_np_s, p_pred_s.numpy(), 'r--', lw=2, label=f'rippl (L2={l2_p_s:.3e})')
axes[1].set_title('Pressure p'); axes[1].set_xlabel('x'); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.suptitle('Stokes 1D Couette — rippl (fixed dim collision)', fontsize=13)
plt.tight_layout()
plt.savefig('stokes1d_result.png', dpi=150, bbox_inches='tight')
plt.show()

## BENCHMARK 4 — DeepXDE Comparison (Heat 1D)
Same problem, same architecture, default DeepXDE settings.

In [ ]:
# CELL 10 — DeepXDE heat equation
try:
    import deepxde as dde
    import numpy as np

    def pde_dde(x, y):
        dy_t = dde.grad.jacobian(y, x, i=0, j=1)
        dy_xx = dde.grad.hessian(y, x, i=0, j=0)
        return dy_t - 0.1 * dy_xx

    geom = dde.geometry.Interval(0, 1)
    td = dde.geometry.TimeDomain(0, 1)
    gt = dde.geometry.GeometryXTime(geom, td)
    bc = dde.icbc.DirichletBC(gt, lambda x: 0, lambda x, on_b: on_b)
    ic = dde.icbc.IC(gt, lambda x: np.sin(np.pi*x[:,0:1]), lambda x, on_i: on_i)
    data = dde.data.TimePDE(gt, pde_dde, [bc, ic], num_domain=10000, num_boundary=2000, num_initial=2000)
    net = dde.nn.FNN([2]+[50]*5+[1], 'tanh', 'Glorot normal')
    model_dde = dde.Model(data, net)

    t0_dde = time.time()
    model_dde.compile('adam', lr=5e-4)
    model_dde.train(iterations=8000, display_every=2000)
    model_dde.compile('L-BFGS')
    model_dde.train(display_every=500)
    train_time_dde = time.time() - t0_dde

    x_np_e = X.cpu().numpy().flatten()
    t_np_e = T.cpu().numpy().flatten()
    xt_e = np.stack([x_np_e, t_np_e], axis=1)
    u_dde_pred = model_dde.predict(xt_e).reshape(200, 200)
    u_true_np = u_true_heat.numpy()
    l2_dde = float(np.sqrt(((u_dde_pred - u_true_np)**2).sum()) / np.sqrt((u_true_np**2).sum()))

    print(f'\n=== DeepXDE HEAT 1D === L2={l2_dde:.6e}  Time={train_time_dde:.1f}s')
    print(f'rippl vs DeepXDE: {l2_heat:.4e} vs {l2_dde:.4e}  rippl better: {l2_heat < l2_dde}')
    results['heat_deepxde'] = {'l2': l2_dde, 'time': train_time_dde, 'rippl_better': l2_heat < l2_dde}
except Exception as e:
    print(f'DeepXDE skipped: {e}')
    results['heat_deepxde'] = {'status': 'skipped', 'reason': str(e)}

## BENCHMARK 5 — NTK Adaptive vs Fixed Loss Weighting
Same heat equation, same architecture. Fixed 100x vs gradient-norm NTK.

In [ ]:
# CELL 11 — NTK vs fixed weights
N_EPOCHS_NTK = 5000

def eval_heat_model(model):
    x_e2 = torch.linspace(0,1,100,device=device)
    t_e2 = torch.linspace(0,1,100,device=device)
    X2, T2 = torch.meshgrid(x_e2, t_e2, indexing='ij')
    ce = torch.stack([X2.flatten(), T2.flatten()], dim=1)
    with torch.no_grad():
        up = model(ce).reshape(100,100).cpu()
    ut = (torch.sin(math.pi*X2) * torch.exp(-alpha*math.pi**2*T2)).cpu()
    return relative_l2(up, ut).item()

# Fixed weights
print(f'Training fixed weights ({N_EPOCHS_NTK} epochs)...')
m_fixed = MLP(2, 1, hidden=50, layers=5).to(device)
opt_f = torch.optim.Adam(m_fixed.parameters(), lr=5e-4)
losses_fixed = []
for epoch in range(N_EPOCHS_NTK):
    opt_f.zero_grad()
    loss, res, l_ic, l_bc = heat_loss(m_fixed, coords_col)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(m_fixed.parameters(), 1.0)
    opt_f.step()
    losses_fixed.append(res.item())
l2_fixed = eval_heat_model(m_fixed)
print(f'Fixed L2: {l2_fixed:.4e}')

# NTK gradient norm weighting
print(f'Training NTK weights ({N_EPOCHS_NTK} epochs)...')
m_ntk = MLP(2, 1, hidden=50, layers=5).to(device)
opt_n = torch.optim.Adam(m_ntk.parameters(), lr=5e-4)
w_res, w_ic, w_bc = 1.0, 1.0, 1.0
losses_ntk = []

for epoch in range(N_EPOCHS_NTK):
    opt_n.zero_grad()
    c = coords_col.clone().requires_grad_(True)
    u = m_ntk(c)
    du = torch.autograd.grad(u.sum(), c, create_graph=True)[0]
    u_t = du[:, 1:2]; u_x = du[:, 0:1]
    u_xx = torch.autograd.grad(u_x.sum(), c, create_graph=True)[0][:, 0:1]
    res_t = (u_t - alpha*u_xx).pow(2).mean()
    ic_t = F.mse_loss(m_ntk(coords_ic), u_ic)
    bc_t = F.mse_loss(m_ntk(coords_bcl), u_bc) + F.mse_loss(m_ntk(coords_bcr), u_bc)
    total = w_res*res_t + w_ic*ic_t + w_bc*bc_t
    total.backward(retain_graph=True)

    # NTK weight update every 100 epochs
    if epoch % 100 == 0 and epoch > 0:
        total_gn = sum(p.grad.norm().item()**2 for p in m_ntk.parameters() if p.grad is not None)**0.5
        for name, lt in [('res', res_t), ('ic', ic_t), ('bc', bc_t)]:
            opt_n.zero_grad()
            lt.backward(retain_graph=True)
            gn = sum(p.grad.norm().item()**2 for p in m_ntk.parameters() if p.grad is not None)**0.5
            ratio = float(np.clip(total_gn / (gn + 1e-8), 0.01, 100.0))
            if name == 'res': w_res = 0.9*w_res + 0.1*ratio
            elif name == 'ic': w_ic = 0.9*w_ic + 0.1*ratio
            else: w_bc = 0.9*w_bc + 0.1*ratio
        opt_n.zero_grad()
        (w_res*res_t + w_ic*ic_t + w_bc*bc_t).backward()

    torch.nn.utils.clip_grad_norm_(m_ntk.parameters(), 1.0)
    opt_n.step()
    losses_ntk.append(res_t.item())

l2_ntk = eval_heat_model(m_ntk)
print(f'NTK L2: {l2_ntk:.4e}')
print(f'NTK better than fixed: {l2_ntk < l2_fixed}  (ratio: {l2_fixed/l2_ntk:.2f}x)')
results['ntk_vs_fixed'] = {'fixed_l2': l2_fixed, 'ntk_l2': l2_ntk, 'ntk_better': bool(l2_ntk < l2_fixed)}

In [ ]:
# CELL 12 — NTK convergence plot
plt.figure(figsize=(10, 4))
plt.semilogy(losses_fixed, alpha=0.8, label=f'Fixed 100x (L2={l2_fixed:.3e})', color='tomato')
plt.semilogy(losses_ntk, alpha=0.8, label=f'NTK adaptive (L2={l2_ntk:.3e})', color='steelblue')
plt.xlabel('Epoch'); plt.ylabel('PDE Residual')
plt.title('NTK Adaptive vs Fixed Loss Weighting — Heat 1D')
plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('ntk_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## FINAL RESULTS

In [ ]:
# CELL 13 — Summary table
print('=' * 60)
print('RIPPL GPU BENCHMARK RESULTS v2 (post-fix)')
print('=' * 60)

pf = results['preflight']
print(f"Pre-flight sanity:  residual={pf['physics_sanity']['residual']:.2e}  {pf['physics_sanity']['status']}")
print(f"Laplacian fix:      error={pf['laplacian']['error']:.2e}  {pf['laplacian']['status']}")
print(f"Pressure grad fix:  error={pf['pressure_gradient']['error']:.2e}  {pf['pressure_gradient']['status']}")
print('-' * 60)

h = results['heat_1d']
print(f"Heat 1D (rippl):    L2={h['l2']:.4e}  {h['status']}  {h['time']:.0f}s")

if 'heat_deepxde' in results and results['heat_deepxde'].get('status') != 'skipped':
    d = results['heat_deepxde']
    print(f"Heat 1D (DeepXDE):  L2={d['l2']:.4e}  {d['time']:.0f}s  rippl_better={d['rippl_better']}")
else:
    print('Heat 1D (DeepXDE):  skipped')

w = results['wave_1d']
print(f"Wave 1D (causal):   L2={w['l2']:.4e}  {w['status']}  {w['time']:.0f}s")

s = results['stokes_1d']
print(f"Stokes 1D:          u={s['l2_u']:.4e}  p={s['l2_p']:.4e}  {s['status']}  {s['time']:.0f}s")

n = results['ntk_vs_fixed']
print(f"NTK vs Fixed:       ntk={n['ntk_l2']:.4e}  fixed={n['fixed_l2']:.4e}  ntk_better={n['ntk_better']}")

print('=' * 60)
passed = sum(1 for k, v in results.items() if v.get('status') == 'PASS')
total = sum(1 for k, v in results.items() if 'status' in v)
print(f'{passed}/{total} benchmarks PASSED')

with open('benchmark_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print('Saved benchmark_results.json')